# Phase 5 — Model Engineering

Builds and validates the configurable project YOLO11s architecture before the single approved training experiment. This notebook does not start full training.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
import yaml
from ultralytics.data.dataset import YOLODataset
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.model_engineering import (
    ProjectYOLO11s,
    benchmark_memory,
    detect_hardware,
    load_yaml,
    model_summary,
    recommend_hyperparameters,
    serialize_hardware,
)

hardware = detect_hardware()
model_config = ROOT / 'configs' / 'model.yaml'
training_config = ROOT / 'configs' / 'training.yaml'
experiment_config = load_yaml(ROOT / 'configs' / 'experiment.yaml')['experiment']
training = load_yaml(training_config)
logger = configure_logger(ROOT / experiment_config['logs_directory'], 'model_engineering')
reports_dir = ROOT / experiment_config['reports_directory']
data_yaml = ROOT / training['training']['dataset']
data = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
model = ProjectYOLO11s(model_config, training_config, nc=data['nc'], verbose=False)
summary = model_summary(model, training['training']['image_size'], hardware.device, load_yaml(model_config)['cbam']['locations'], training['loss']['focal'])
benchmark = benchmark_memory(
    model,
    hardware.device,
    training['training']['image_size'],
    experiment_config['benchmark_batch_sizes'],
    experiment_config['benchmark_iterations'],
    experiment_config['benchmark_warmup_iterations'],
)
recommendations = recommend_hyperparameters(hardware, benchmark)
dataset = YOLODataset(img_path=str(ROOT / experiment_config['final_dataset'] / 'train/images'), imgsz=training['training']['image_size'], augment=False, cache=False, data=data)
batch = dataset.collate_fn([dataset[0], dataset[1]])
batch = {key: (value.to(hardware.device) if isinstance(value, torch.Tensor) else value) for key, value in batch.items()}
batch['img'] = batch['img'].float() / 255
model = model.to(hardware.device).train()
forward_output = model(batch['img'])
loss, loss_components = model(batch)
loss.sum().backward()
model.zero_grad(set_to_none=True)
smoke_test = {
    'dataset_images': len(dataset),
    'batch_size': int(batch['img'].shape[0]),
    'input_shape': list(batch['img'].shape),
    'forward_output_keys': list(forward_output),
    'loss': float(loss.sum().detach().cpu()),
    'loss_components': {key: float(value.detach().cpu()) for key, value in loss_components.items()},
    'runtime_errors': 0,
}
stamp = utc_timestamp()
hardware_path = write_json(reports_dir / f'hardware_report_{stamp}.json', serialize_hardware(hardware))
summary_path = write_json(reports_dir / f'model_summary_{stamp}.json', summary)
benchmark_path = write_json(reports_dir / f'memory_benchmark_{stamp}.json', {'hardware': serialize_hardware(hardware), 'results': benchmark})
recommendation_path = write_json(reports_dir / f'hyperparameter_recommendations_{stamp}.json', recommendations)
readiness_path = write_json(reports_dir / f'training_readiness_{stamp}.json', {'hardware_report': str(hardware_path), 'model_summary': str(summary_path), 'memory_benchmark': str(benchmark_path), 'recommendations': str(recommendation_path), 'smoke_test': smoke_test, 'full_training_started': False, 'approval_required': True})
logger.info('Hardware: %s', serialize_hardware(hardware))
logger.info('Memory benchmark: %s', benchmark)
logger.info('Smoke test completed without runtime errors.')
print({'hardware': serialize_hardware(hardware), 'model_summary': summary, 'recommendations': recommendations, 'smoke_test': smoke_test, 'training_readiness': str(readiness_path)})

Fast image access ✅ (ping: 0.0±0.0 ms, read: 632.5±247.9 MB/s, size: 179.0 KB)


Scanning /Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels.cache... 105475 images, 9032 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 105475/105475 7.0Git/s 0.0s

/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/00000389_jpg.rf.bd357afc09858f6968f9780e0f780eae.jpg: 1 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/01736917d1d06920_jpg.rf.3aa46698efa6b77a52f71019132b51e3.jpg: 1 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/02792cd530f2d505_jpg.rf.a42d460416bad934d2eb2211fae267bf.jpg: 1 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/0a8ab3f65ad3b6a6_jpg.rf.30dd35fe4bcd4a14e2994059747514ef.jpg: 1 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/0c49a929b8ca1778_jpg.rf.24670915896f1b4fb377cdb12637bff0.jpg: 1 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/images/1-177-_jpg.rf.69a317a9994241354ed8cf6c776a0ecf.jpg: 2 duplicate labels removed
/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/i

{'hardware': {'device': 'mps', 'backend': 'mps', 'chip': 'Apple M4 Pro', 'total_memory_gb': 24.0, 'cpu_cores': 12, 'recommended_workers': 6, 'torch_version': '2.6.0'}, 'model_summary': {'total_parameters': 9479173, 'trainable_parameters': 9479157, 'gflops': 21.324551999999997, 'image_size': 640, 'device': 'mps', 'cbam_locations': ['P3', 'P4', 'P5'], 'loss_configuration': {'enabled': True, 'gamma': 2.0, 'alpha': 0.25}}, 'recommendations': {'model': 'YOLO11s with CBAM at P3, P4, and P5', 'epochs': 120, 'freeze_epochs': 5, 'unfreeze_epochs': 115, 'batch_size': 32, 'image_size': 640, 'learning_rate': 0.003, 'optimizer': 'AdamW', 'weight_decay': 0.0005, 'momentum': 0.9, 'scheduler': 'cosine', 'warmup_epochs': 3, 'workers': 6, 'cache_mode': 'disk', 'amp': True, 'early_stopping_patience': 25, 'checkpoint_frequency': 1, 'validation_frequency': 1, 'benchmark_basis': {'batch_size': 32, 'status': 'stable', 'peak_memory_mb': 15885.15625, 'images_per_second': 27.061101326658218, 'error': None}, 'ap

In [2]:
import math

baseline = ProjectYOLO11s(model_config, training_config, nc=data['nc'], verbose=False, cbam_enabled=False)
baseline_summary = model_summary(baseline, training['training']['image_size'], hardware.device, [], training['loss']['focal'])
additional_parameters = summary['total_parameters'] - baseline_summary['total_parameters']
additional_gflops = summary['gflops'] - baseline_summary['gflops']
cbam_review = {
    'enabled': load_yaml(model_config)['cbam']['enabled'],
    'locations': load_yaml(model_config)['cbam']['locations'],
    'selection_rationale': {
        'P3': 'Retains fine-grained spatial detail for small handheld threats and distant persons.',
        'P4': 'Balances semantic context and localization for medium-scale targets.',
        'P5': 'Adds global channel/spatial context for large persons, fire, and firearms.',
    },
    'additional_parameters': additional_parameters,
    'additional_gflops': additional_gflops,
    'estimated_compute_overhead_percent': additional_gflops / baseline_summary['gflops'] * 100,
    'inference_overhead_assessment': 'Expected to be low because the measured FLOPs increase is below one tenth of one percent; end-to-end latency must be measured after training.',
    'expected_performance_gain': 'Attention may improve recall on cluttered and multi-scale threats, but the gain is an unverified hypothesis until compared using identical validation conditions.',
    'alternative_placement': 'If latency becomes critical, retain P3 and P4 CBAM first; P5 is the first location to ablate because it has the highest channel count.',
}
focal_review = {
    'configuration': training['loss']['focal'],
    'class_distribution': {'Handheld_Weapon': 61168, 'Explosive': 25288, 'Fire_Smoke': 77860, 'Firearm': 117069, 'Person': 15173},
    'review': 'Gamma 2.0 and alpha 0.25 are appropriate conservative starting values after Person-only augmentation. Person remains the minority class, but its class weight has been reduced after augmentation; increasing gamma or alpha before baseline validation risks suppressing easy positives and destabilizing convergence.',
    'recommendation': 'Keep gamma 2.0 and alpha 0.25 for the single approved run unless an approved ablation replaces the single-run constraint.',
}
best_benchmark = recommendations['benchmark_basis']
steps_per_epoch = math.ceil(smoke_test['dataset_images'] / best_benchmark['batch_size'])
train_seconds = smoke_test['dataset_images'] / best_benchmark['images_per_second']
time_estimation = {
    'steps_per_epoch': steps_per_epoch,
    'synthetic_training_images_per_second': best_benchmark['images_per_second'],
    'synthetic_training_duration_minutes_per_epoch': train_seconds / 60,
    'synthetic_training_duration_hours_for_120_epochs': train_seconds * recommendations['epochs'] / 3600,
    'validation_time': 'Not estimated from synthetic training throughput. Measure a validation-only pass before the approved training run rather than guessing.',
    'qualification': 'These values use measured synthetic forward/backward throughput and exclude disk I/O, validation, checkpointing, and data-loader variability.',
}
lr_analysis = {
    'recommended_initial_learning_rate': recommendations['learning_rate'],
    'optimizer': recommendations['optimizer'],
    'reasoning': 'The measured batch size of 32 supports a moderate AdamW learning rate. A 0.003 starting rate is intentionally below aggressive large-batch detection defaults to protect stability when CBAM and focal classification modulation are both active.',
    'guardrail': 'Do not raise the rate without an approved learning-rate range test; the one-run constraint prevents empirical tuning after training begins.',
}
checkpoint_strategy = {
    'save_every_epoch': True,
    'ultralytics_settings': {'save': True, 'save_period': 1, 'resume': True},
    'artifacts': ['last.pt every epoch', 'best.pt when validation fitness improves'],
    'resume_behavior': 'Use Ultralytics resume from last.pt. The checkpoint stores model, optimizer, scheduler, AMP scaler, epoch, and best fitness state.',
    'workflow': ['Verify writable run directory before launch.', 'Preserve the complete run directory.', 'Resume only the same approved configuration and final dataset revision.', 'Never overwrite best.pt or last.pt manually.'],
}
overfitting_review = {
    'dataset_size': '130,245 source images plus 1,222 Person-only training augmentations is substantial, which lowers but does not remove overfitting risk.',
    'person_augmentation': 'The 60% minority-target strategy is conservative and avoids equalizing classes. Restricting augmentation to Person-only images prevents synthetic duplication of other classes.',
    'class_weights': 'Person remains upweighted. Combining class weights with focal loss can overemphasize rare hard examples; monitor Person precision versus recall.',
    'cbam': 'CBAM adds limited capacity and compute, but may overfit if training is extended after validation plateaus.',
    'freeze_strategy': 'Five frozen epochs reduce early destructive updates if pretrained weights are used. Confirm pretrained-weight loading before final launch.',
    'regularization': 'AdamW weight decay 0.0005, cosine schedule, three warmup epochs, per-epoch validation, and patience 25 are appropriate controls.',
    'critical_risk': 'Ultralytics dataset loading reported duplicate label removal in source label files. The project validator did not flag duplicates. Do not begin the one training run until you approve a non-destructive duplicate-label audit and a new derived final dataset revision if needed.',
}
final_engineering_review = {
    'pipeline': ['original preserved', 'invalid pairs excluded in cleaned_v1', 'classes merged in processed_v1', 'Person-only augmentation in final_dataset_v1', 'YOLO11s CBAM with configurable focal loss'],
    'strengths': ['source datasets remain immutable', 'final label coordinates and class IDs validate', 'MPS benchmark completed through batch 32', 'smoke test completed with zero runtime errors'],
    'bottlenecks': ['MPS throughput is approximately 27 images/s in synthetic forward/backward benchmarking', 'validation and checkpoint I/O are not included in the benchmark'],
    'risks': [overfitting_review['critical_risk'], 'Single-run policy prevents normal empirical ablation of CBAM, focal loss, and learning rate.'],
    'approval_gate': 'Do not start training until the duplicate-label risk, pretrained initialization, and recommended configuration are explicitly approved.',
}
stamp = utc_timestamp()
cbam_path = write_json(reports_dir / f'cbam_review_{stamp}.json', cbam_review)
focal_path = write_json(reports_dir / f'focal_loss_review_{stamp}.json', focal_review)
lr_path = write_json(reports_dir / f'learning_rate_analysis_{stamp}.json', lr_analysis)
overfitting_path = write_json(reports_dir / f'overfitting_risk_analysis_{stamp}.json', overfitting_review)
time_path = write_json(reports_dir / f'training_time_estimation_{stamp}.json', time_estimation)
checkpoint_path = write_json(reports_dir / f'checkpoint_strategy_{stamp}.json', checkpoint_strategy)
final_review_path = write_json(reports_dir / f'final_model_engineering_report_{stamp}.json', final_engineering_review)
print({'cbam_review': str(cbam_path), 'focal_review': str(focal_path), 'learning_rate_analysis': str(lr_path), 'overfitting_review': str(overfitting_path), 'time_estimation': str(time_path), 'checkpoint_strategy': str(checkpoint_path), 'final_review': str(final_review_path)})

{'cbam_review': '/Users/mohamedtarek/weapon130/reports/cbam_review_20260723T103750Z.json', 'focal_review': '/Users/mohamedtarek/weapon130/reports/focal_loss_review_20260723T103750Z.json', 'learning_rate_analysis': '/Users/mohamedtarek/weapon130/reports/learning_rate_analysis_20260723T103750Z.json', 'overfitting_review': '/Users/mohamedtarek/weapon130/reports/overfitting_risk_analysis_20260723T103750Z.json', 'time_estimation': '/Users/mohamedtarek/weapon130/reports/training_time_estimation_20260723T103750Z.json', 'checkpoint_strategy': '/Users/mohamedtarek/weapon130/reports/checkpoint_strategy_20260723T103750Z.json', 'final_review': '/Users/mohamedtarek/weapon130/reports/final_model_engineering_report_20260723T103750Z.json'}


In [3]:
final_readiness_path = write_json(reports_dir / f'final_training_readiness_{stamp}.json', {
    'hardware_report': str(hardware_path),
    'memory_benchmark_report': str(benchmark_path),
    'model_summary': str(summary_path),
    'cbam_review': str(cbam_path),
    'focal_loss_review': str(focal_path),
    'learning_rate_analysis': str(lr_path),
    'overfitting_risk_analysis': str(overfitting_path),
    'hyperparameter_recommendations': str(recommendation_path),
    'training_time_estimation': str(time_path),
    'checkpoint_strategy': str(checkpoint_path),
    'final_model_engineering_report': str(final_review_path),
    'smoke_test': smoke_test,
    'full_training_started': False,
    'approval_required': True,
})
logger.info('Final model engineering review is ready for approval: %s', final_readiness_path)
print({'final_training_readiness': str(final_readiness_path), 'full_training_started': False})

{'final_training_readiness': '/Users/mohamedtarek/weapon130/reports/final_training_readiness_20260723T103750Z.json', 'full_training_started': False}
